# Disease Inference Using Symptom Vector Representations and a Graph Database

This notebook demonstrates a neuro-symbolic pipeline for ranking candidate diseases based on symptoms reported by the user. The experiment focuses on semantic symptom mapping using vector embeddings and symbolic reasoning over a medical ontology stored in a Neo4j graph database.

## Assumption

This experiment assumes the existence of a preceding NLP layer that has already extracted relevant symptoms from the user's natural language input. Therefore, **the process of extracting symptoms from raw text is not considered here**. Instead, the goal is to evaluate the following steps:

- Semantic mapping of extracted symptoms to ontological concepts using vector embeddings,

- Symbolic disease inference using knowledge from a Neo4j ontological graph.

In [37]:
# [1] No absent symptoms
# symptom_input = {"present": ["headache", "fever", "stiff neck", "vomiting", "seizure"], "absent": []}

# [2] Should  return 'West Nile encephalitis', 	Japanese encephalitis eliminated
# symptom_input = {"present": ["headache", "high fever", "coma", "confusion", "stiff neck"], "absent": ["spastic paralysis"]}

# [3] Should return 'hepatitis E', 'hepatitis' D eliminated
# symptom_input = {"present": ["jaundice", "nausea", "fatigue", "dark urine", "acholic stool"], "absent": ["confusion", "drowsiness"]} 

# [4] Should return 'Ebola virus disease', 'Marburg hemorrhagic fever' eliminated
symptom_input = {"present": ["fever", "headache", "vomiting", "bleeding", "pharynx inflammation"], "absent": ["maculopapular rash", "chills"]}


# Step 1: Vector Embedding

Each extracted symptom is mapped into a continuous vector space using the pre-trained **intfloat/multilingual-e5-large** model. This phase enables semantic comparison between symptoms provided by the user and symptoms defined in the ontology.

Embeddings for ontological symptoms are generated during a preprocessing phase and persisted in the graph database, while embeddings for input symptoms are computed at inference time for each user query.

In [ ]:
from sentence_transformers import SentenceTransformer

#model = SentenceTransformer('all-MiniLM-L6-v2')
#model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

all_input_symptoms = symptom_input["present"] + symptom_input["absent"]
query_embeddings = model.encode(all_input_symptoms)

## Loading Ontological Symbols and Vector Embeddings

In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv(override=True)

neo4j_url      = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        RETURN s.n4sch__label[0] AS label, s.embedding AS embedding
    """)
    ontology_symptoms = [(r["label"], r["embedding"]) for r in result]

print(f"Loaded {len(ontology_symptoms)} ontological symptoms.")

# Step 2: Semantic Symptom Mapping

Cosine similarity is used to compare the vector embeddings of input symptoms against ontological symptoms stored in the Neo4j graph database.
For each input symptom, the ontological term with the highest similarity score is identified and used as its semantic equivalent in the downstream inference process.

## Symptom Matching via Cosine Similarity

Each input symptom embedding is compared against all ontological symptom embeddings using cosine similarity. The ontological term with the highest similarity score is selected as the semantic match. A confidence threshold of **0.9** is applied — symptoms that fall below this threshold are excluded from the inference step to avoid noisy or ambiguous mappings.

Input symptoms are split into two lists based on their kind:
- **Present symptoms** — mapped ontological terms that will be used to find matching diseases,
- **Absent symptoms** — mapped ontological terms that will be used to exclude diseases.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

onto_labels  = [o[0] for o in ontology_symptoms]
onto_vectors = np.array([o[1] for o in ontology_symptoms])

CONFIDENCE_THRESHOLD = 0.9

matched = []
for symptom, q_emb in zip(all_input_symptoms, query_embeddings):
    sims     = cosine_similarity([q_emb], onto_vectors)[0]
    best_idx = sims.argmax()
    matched.append({
        "input_symptom":  symptom,
        "mapped_symptom": onto_labels[best_idx],
        "confidence":     float(sims[best_idx]),
        "kind":           "present" if symptom in symptom_input["present"] else "absent"
    })

print(f"{'USER INPUT':<25} | {'KIND':<8} | {'ONTOLOGICAL TERM':<30} | {'SIMILARITY'}")
print("-" * 85)
for m in matched:
    flag = "" if m["confidence"] > CONFIDENCE_THRESHOLD else " -- UNDER THRESHOLD"
    print(f"{m['input_symptom']:<25} | {m['kind']:<8} | {m['mapped_symptom']:<30} | {m['confidence']:.4f}{flag}")

present_symptoms = [
    m["mapped_symptom"] for m in matched
    if m["kind"] == "present" and m["confidence"] > CONFIDENCE_THRESHOLD
]
absent_symptoms = [
    m["mapped_symptom"] for m in matched
    if m["kind"] == "absent" and m["confidence"] > CONFIDENCE_THRESHOLD
]

print(f"\nPresent: {present_symptoms}")
print(f"Absent:  {absent_symptoms}")

# Step 3: Neo4j Symbolic Inference

Disease ranking is not performed by simple symptom count matching. Instead, a weighted sum of symptoms present in each disease is used. Each symptom carries a pre-computed **Information Content (IC)** weight that reflects how informative that symptom is for differentiating between diseases.

The query sums these weights across all symptoms of a disease that match the user's input (`total_score`). To account for diseases with varying numbers of symptoms, the total score is normalized by dividing by the square root of the disease's total symptom count (`normalized_score`). This achieves a balance between:

1. **Symptom specificity** — symptoms with higher IC contribute more to the score,
2. **Disease complexity** — diseases with many symptoms do not automatically dominate the ranking.

This approach allows diseases with fewer but more informative symptoms to rank higher than diseases with many but less specific symptoms.

## Absent Symptom Filtering

In addition to scoring, the query incorporates **explicit exclusion logic** based on symptoms the user has reported as absent. The inference pipeline executes in the following stages:

1. **Collect all symptoms per disease** — for each disease node, all associated symptoms are retrieved along with the total symptom count,
2. **Identify blocking symptoms** — for each disease, the query checks whether any of its symptoms appear in the absent symptoms list. These are collected as `blocking_symptoms`,
3. **Evaluate filter** — `passed_filter` is set to `true` if the disease has no blocking symptoms, and `false` otherwise. Crucially, **excluded diseases are still scored and returned** — they are not dropped from results — so that the reasoning layer can explain why they were ruled out,
4. **Match present symptoms** — only symptoms from the user's present list are matched and scored,
5. **Apply minimum match threshold** — diseases with fewer than `$min_match` present symptom matches are discarded entirely,
6. **Compute coverage metrics** — `missing_symptoms` lists the disease symptoms not found in the user's present input, providing a gap analysis per candidate disease.

Results are ordered by `passed_filter DESC` (included diseases first) and `normalized_score DESC` within each group.

## Query Output Format

Each record returned by the query represents one candidate disease. Below is an example entry with field descriptions:
```json
{
  "disease":             "nonparalytic poliomyelitis",
  "uri":                 "http://purl.obolibrary.org/obo/DOID_4986",
  "passed_filter":       true,
  "blocking_symptoms":   [],
  "matched_symptoms":    ["headache", "pharynx inflammation", "fever", "vomiting"],
  "missing_symptoms":    ["fatigue", "stiff neck"],
  "match_count":         4,
  "total_symptom_count": 6,
  "total_score":         21.386,
  "normalized_score":    8.731
}
```

| Field | Type | Description |
|---|---|---|
| `disease` | `string` | Disease label from the ontology |
| `uri` | `string` | Unique ontological URI identifying the disease |
| `passed_filter` | `boolean` | `true` if no absent symptoms were found in the disease profile, `false` if the disease was excluded |
| `blocking_symptoms` | `list[string]` | Absent symptoms reported by the user that are part of this disease's profile — empty if `passed_filter` is `true` |
| `matched_symptoms` | `list[string]` | Present symptoms reported by the user that were found in this disease's profile |
| `missing_symptoms` | `list[string]` | Symptoms in the disease profile that were not reported as present by the user |
| `match_count` | `integer` | Number of matched present symptoms |
| `total_symptom_count` | `integer` | Total number of symptoms associated with the disease in the ontology |
| `total_score` | `float` | Sum of IC weights of all matched symptoms |
| `normalized_score` | `float` | `total_score / sqrt(total_symptom_count)` — score normalized for disease complexity |

In [ ]:
MIN_MATCH = 2
TOP_K     = 5
TOP_EXCLUDED = 5
HAS_SYMPTOM = "http://purl.obolibrary.org/obo/RO_0002452"

from pathlib import Path
CYPHER_QUERY_PATH = Path.cwd() / "tests" / "inference-and-scoring-test" / "queries" / "cypher-query.cyp"

with open(CYPHER_QUERY_PATH) as f:
    QUERY = f.read()
print(f"Cypher query loaded successfully.")

In [ ]:
total_input_symptoms = len(present_symptoms)

with driver.session() as session:
    r = session.run(
        QUERY,
        has_symptom=HAS_SYMPTOM,
        present_symptoms=present_symptoms,
        absent_symptoms=absent_symptoms,
        min_match=MIN_MATCH
    )
    records = list(r)
included = []
excluded = []

for rec in records:
    disease_cov = round(rec['match_count'] / rec['total_symptom_count'] * 100, 1)
    input_cov   = round(rec['match_count'] / total_input_symptoms * 100, 1)
    print(rec)
    entry = {
        "disease":             rec["disease"],
        "uri":                 rec["uri"],
        "matched_symptoms":    rec["matched_symptoms"],
        "missing_symptoms":    rec["missing_symptoms"],
        "match_count":         rec["match_count"],
        "total_symptom_count": rec["total_symptom_count"],
        "normalized_score":    rec["normalized_score"],
        "disease_coverage":    disease_cov,
        "input_coverage":      input_cov,
    }

    if rec["passed_filter"]:
        included.append(entry)
    else:
        excluded.append({
            **entry,
            "blocking_symptoms": rec["blocking_symptoms"]
        })

included = sorted(included, key=lambda x: x["normalized_score"], reverse=True)[:TOP_K]
excluded = sorted(excluded, key=lambda x: x["normalized_score"], reverse=True)[:TOP_EXCLUDED]

In [ ]:
print(f"\n{'INCLUDED DISEASES':}")
print(f"{'DISEASE':<40} | {'SCORE':<6} | {'MATCH':<5} | {'D.COV':<7} | {'I.COV':<7} | {'MATCHED'} | {'MISSING'}")
print("-" * 200)

for d in included:
    print(f"{d['disease']:<40} | "
          f"{d['normalized_score']:.2f}   | "
          f"{d['match_count']:<5} | "
          f"{d['disease_coverage']}%{'':2} | "
          f"{d['input_coverage']}%{'':2} | "
          f"{d['matched_symptoms']} | "
          f"{d['missing_symptoms']}")

In [ ]:
print(f"\n{'EXCLUDED DISEASES (because of absent symptoms)':}")
print(f"{'DISEASE':<40} | {'SCORE':<6} | {'MATCH':<5} | {'BLOCKING SYMPTOMS'}")
print("-" * 120)

for d in excluded:
    print(f"{d['disease']:<40} | "
          f"{d['normalized_score']:.2f}   | "
          f"{d['match_count']:<5} | "
          f"{d['blocking_symptoms']}")

## Embedding Model Evaluation

The results and methodology of the embedding model evaluation are described in detail in **embedding-test**.

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "embeddings-test" / "symptom-full-test.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print(f"{len(test_cases)} test cases loaded.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import json
import os

# model = SentenceTransformer('all-MiniLM-L6-v2')
# model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

all_embeddings = []
all_metadata = []
matched = {}

for case in test_cases:
    symptoms = case.get("extracted_symptoms", [])
    case_id = case.get("id", "unknown")

    if not symptoms:
        continue

    if case_id not in matched:
        matched[case_id] = []

    for symptom in symptoms:
        embedding = model.encode(symptom.strip(), normalize_embeddings=True)
        onto_labels = [o[0] for o in ontology_symptoms]
        onto_vectors = np.array([o[1] for o in ontology_symptoms])

        sims = cosine_similarity([embedding], onto_vectors)[0]
        best_idx = sims.argmax()

        matched[case_id].append({
            "input_symptom": symptom,
            "mapped_symptom": onto_labels[best_idx],
            "confidence": float(sims[best_idx])
        })

output_dir = "tests/embeddings-test/results"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "embedding-test-3.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(matched, f, indent=2, ensure_ascii=False)

print(f"Results written at: {output_path}")
print(f"Total cases: {len(matched)}")

In [ ]:
def evaluate_embedding_layer(matched_results):
    usable_threshold = 0.90

    def empty_stats():
        return {
            "total": 0,
            "exact": 0,
            "usable": 0,
            "fail": 0,
            "conf_sum": 0.0,
            "fail_cases": [],
            "non_exact_usable": []
        }

    categories = {
        "EXACT":  empty_stats(),
        "SYN":    empty_stats(),
        "LLM":   empty_stats(),
        "SUM": empty_stats(),
    }

    for case_id, mappings in matched_results.items():
        if not mappings:
            continue

        best = mappings[0]
        conf      = best["confidence"]
        input_sym = best["input_symptom"]
        mapped_sym = best["mapped_symptom"]
        is_exact  = (input_sym.strip().lower() == mapped_sym.strip().lower())

        if case_id.startswith("EXACT"):
            cat = "EXACT"
        elif case_id.startswith("SYN"):
            cat = "SYN"
        elif case_id.startswith("LLM"):
            cat = "LLM"
        else:
            cat = "SUM"

        for key in [cat, "SUM"]:
            s = categories[key]
            s["total"]    += 1
            s["conf_sum"] += conf

            if is_exact:
                s["exact"] += 1

            if conf >= usable_threshold:
                s["usable"] += 1
                if not is_exact:
                    s["non_exact_usable"].append((case_id, input_sym, mapped_sym, conf))
            else:
                s["fail"] += 1
                s["fail_cases"].append((case_id, input_sym, mapped_sym, conf))

    def print_category(name, s):
        total = s["total"]
        if total == 0:
            print(f"\n[{name}] — No data available.")
            return

        exact_rate   = (s["exact"]  / total) * 100
        usable_rate  = (s["usable"] / total) * 100
        bad_rate     = (s["fail"]   / total) * 100
        avg_conf     = s["conf_sum"] / total

        is_summary = (name == "SUM")
        sep = "═" * 60 if is_summary else "─" * 60

        print(f"\n{sep}")
        if is_summary:
            print(f"{'  SUMMARY — ALL CATEGORIES  ':^60}")
        else:
            desc = {"EXACT": "180 terms from ontology", "SYN": "70 synonyms", "LLM": "llm cases"}.get(name, "")
            print(f"{'  CATEGORY: ' + name + ' (' + str(total) + ' tests' + ((' — ' + desc) if desc else '') + ')  ':^60}")
        print(sep)

        print(f"  {'Metric':<32} | {'Value':>9} | Status")
        print(f"  {'─'*32}─+─{'─'*9}─+─{'─'*6}")

        print(f"  {'Exact Match Rate':<32} | {exact_rate:>8.1f}%")
        print(f"  {'Usable Match Rate  (conf ≥ 0.90)':<32} | {usable_rate:>8.1f}%")
        print(f"  {'Average Confidence':<32} | {avg_conf:>9.4f}")
        print(f"  {'Bad Match Rate     (conf < 0.90)':<32} | {bad_rate:>8.1f}%")
        print(f"  {'─'*55}")
        print(f"  Exact: {s['exact']}  |  Usable (non-exact): {len(s['non_exact_usable'])}  |  Fail: {s['fail']}  |  Total: {total}")

    for cat in ["EXACT", "SYN", "LLM"]:
        print_category(cat, categories[cat])

    print_category("SUM", categories["SUM"])

    return categories

In [ ]:
results = evaluate_embedding_layer(matched)

# Inference Layer Evaluation

The results and methodology of the inference and scoring layer evaluation are described in detail in **inference-test**.

## Cypher query

In [ ]:
MIN_MATCH    = 2
TOP_K        = 5
TOP_EXCLUDED = 5
HAS_SYMPTOM  = "http://purl.obolibrary.org/obo/RO_0002452"

from pathlib import Path

BASE_DIR = Path.cwd()

CYPHER_QUERY_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "queries" / "cypher-query.cyp"

with open(CYPHER_QUERY_PATH) as f:
    QUERY = f.read()

print(f"Cypher query loaded successfully.")

## Test Data (Ground Truth)

The test data is exported from the DOID and SYMP ontology and contains a list of diseases, each associated with the symptoms defined for it in the ontology. Each symptom is wrapped in its own list due to the export format from the ontology.

```json
{
  "disease":   ["<disease name>"],
  "symptoms":  [
    ["<symptom 1>"],
    ["<symptom 2>"],
    ["<symptom n>"],
  ]
}
```

In [ ]:
import json

GROUND_TRUTH_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "symptom-disease-test-data.json"

with open(GROUND_TRUTH_PATH) as f:
    raw = json.load(f)

ground_truth = [
    {
        "disease":  item["disease"][0],
        "symptoms": [s[0] for s in item["symptoms"]]
    }
    for item in raw
    if len(item["symptoms"]) >= MIN_MATCH
]

print(f"Loaded {len(ground_truth)} test cases (diseases with >= {MIN_MATCH} symptoms).")

In [ ]:
def run_query(session, present_symptoms, absent_symptoms=None):
    query_result = session.run(
        QUERY,
        has_symptom=HAS_SYMPTOM,
        present_symptoms=present_symptoms,
        absent_symptoms=absent_symptoms or [],
        min_match=MIN_MATCH
    )
    return list(query_result)


def evaluate(ground_truth, session, drop_n=0):
    hit1 = hit3 = hit5 = total = no_results = 0
    details = []

    for item in ground_truth:
        disease  = item["disease"]
        symptoms = item["symptoms"]

        if len(symptoms) <= drop_n:
            continue

        present_symptoms = symptoms[:-drop_n] if drop_n > 0 else symptoms
        records  = run_query(session, present_symptoms)

        included = [r for r in records if r["passed_filter"]]
        excluded = [r for r in records if not r["passed_filter"]]

        ranked = [r["disease"] for r in included]
        total += 1

        if not ranked:
            no_results += 1
            details.append({
                "disease":          disease,
                "present_symptoms": present_symptoms,
                "ranked":           [],
                "rank":             None,
                "hit@1":            False,
                "hit@3":            False,
                "hit@5":            False,
                "excluded":         []
            })
            continue

        rank = ranked.index(disease) + 1 if disease in ranked else None
        h1 = rank == 1     if rank else False
        h3 = rank <= 3     if rank else False
        h5 = rank <= TOP_K if rank else False

        if h1: hit1 += 1
        if h3: hit3 += 1
        if h5: hit5 += 1

        details.append({
            "disease":          disease,
            "present_symptoms": present_symptoms,
            "ranked":           ranked[:TOP_K],
            "rank":             rank,
            "hit@1":            h1,
            "hit@3":            h3,
            "hit@5":            h5,
            "top_score":        included[0]["total_score"] if included else None,
            "excluded":         [
                {
                    "disease":           r["disease"],
                    "blocking_symptoms": r["blocking_symptoms"],
                    "match_count":       r["match_count"]
                }
                for r in excluded[:TOP_EXCLUDED]
            ]
        })

    return {
        "hit@1":      hit1,
        "hit@3":      hit3,
        "hit@5":      hit5,
        "total":      total,
        "no_results": no_results,
        "details":    details
    }

# Full test

All symptoms associated with a disease are provided as input. This represents the ideal scenario where complete symptom information is available.

In [ ]:
with driver.session() as session:
    full = evaluate(ground_truth, session, drop_n=0)

t = full["total"]
print(f"{'='*50}")
print(f"FULL MATCH (all symptoms)")
print(f"{'='*50}")
print(f"Total test cases : {t}")
print(f"No results       : {full['no_results']}")
print(f"Hit@1            : {full['hit@1']}/{t}  ({full['hit@1']/t*100:.1f}%)")
print(f"Hit@3            : {full['hit@3']}/{t}  ({full['hit@3']/t*100:.1f}%)")
print(f"Hit@5            : {full['hit@5']}/{t}  ({full['hit@5']/t*100:.1f}%)")

import json
from datetime import datetime

RESULTS_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "results" / "inference-full-test.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "min_match": MIN_MATCH,
        "top_k":     TOP_K,
        "drop_n":    0
    },
    "summary": {
        "total":      full["total"],
        "no_results": full["no_results"],
        "hit@1":      full["hit@1"],
        "hit@3":      full["hit@3"],
        "hit@5":      full["hit@5"],
        "hit@1_pct":  round(full["hit@1"] / full["total"] * 100, 1),
        "hit@3_pct":  round(full["hit@3"] / full["total"] * 100, 1),
        "hit@5_pct":  round(full["hit@5"] / full["total"] * 100, 1),
    },
    "details": full["details"]
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

# Partial (Drop=1)

One symptom is removed from each disease's symptom list before querying. This simulates a realistic scenario where a patient fails to report one of their symptoms.

In [ ]:
with driver.session() as session:
    full = evaluate(ground_truth, session, drop_n=1)

t = full["total"]
print(f"{'='*50}")
print(f"PARTIAL MATCH (1 symptom dropped)")
print(f"{'='*50}")
print(f"Total test cases : {t}")
print(f"No results       : {full['no_results']}")
print(f"Hit@1            : {full['hit@1']}/{t}  ({full['hit@1']/t*100:.1f}%)")
print(f"Hit@3            : {full['hit@3']}/{t}  ({full['hit@3']/t*100:.1f}%)")
print(f"Hit@5            : {full['hit@5']}/{t}  ({full['hit@5']/t*100:.1f}%)")

import json
from datetime import datetime

RESULTS_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "results" / "inference-partial1-test.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "min_match": MIN_MATCH,
        "top_k":     TOP_K,
        "drop_n":    1
    },
    "summary": {
        "total":      full["total"],
        "no_results": full["no_results"],
        "hit@1":      full["hit@1"],
        "hit@3":      full["hit@3"],
        "hit@5":      full["hit@5"],
        "hit@1_pct":  round(full["hit@1"] / full["total"] * 100, 1),
        "hit@3_pct":  round(full["hit@3"] / full["total"] * 100, 1),
        "hit@5_pct":  round(full["hit@5"] / full["total"] * 100, 1),
    },
    "details": full["details"]
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

# Partial (Drop=2)

Two symptoms are removed from each disease's symptom list before querying. This represents a more challenging scenario with greater symptom incompleteness, further stress-testing the robustness of the ranking.

In [ ]:
with driver.session() as session:
    full = evaluate(ground_truth, session, drop_n=2)

t = full["total"]
print(f"{'='*50}")
print(f"PARTIAL MATCH (2 symptom dropped)")
print(f"{'='*50}")
print(f"Total test cases : {t}")
print(f"No results       : {full['no_results']}")
print(f"Hit@1            : {full['hit@1']}/{t}  ({full['hit@1']/t*100:.1f}%)")
print(f"Hit@3            : {full['hit@3']}/{t}  ({full['hit@3']/t*100:.1f}%)")
print(f"Hit@5            : {full['hit@5']}/{t}  ({full['hit@5']/t*100:.1f}%)")

import json
from datetime import datetime

RESULTS_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "results" / "inference-partial2-test.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "min_match": MIN_MATCH,
        "top_k":     TOP_K,
        "drop_n":    2
    },
    "summary": {
        "total":      full["total"],
        "no_results": full["no_results"],
        "hit@1":      full["hit@1"],
        "hit@3":      full["hit@3"],
        "hit@5":      full["hit@5"],
        "hit@1_pct":  round(full["hit@1"] / full["total"] * 100, 1),
        "hit@3_pct":  round(full["hit@3"] / full["total"] * 100, 1),
        "hit@5_pct":  round(full["hit@5"] / full["total"] * 100, 1),
    },
    "details": full["details"]
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {RESULTS_PATH}")

# Exclusion test

Diseases are queried using their full symptom list, with one additional symptom passed as absent — a symptom belonging to a similar disease but not to the one being tested. This validates the inference layer's ability to correctly filter out diseases based on explicitly reported absent symptoms.

In [ ]:
from collections import defaultdict

def build_exclusion_test_cases(ground_truth, min_shared_symptoms=1, max_cases_per_disease=3):
    disease_symptoms = {item["disease"]: set(item["symptoms"]) for item in ground_truth}

    symptom_index = defaultdict(set)
    for disease, symptoms in disease_symptoms.items():
        for s in symptoms:
            symptom_index[s].add(disease)

    test_cases = []

    for disease, symptoms in disease_symptoms.items():
        candidates = []

        for symptom in symptoms:
            for similar in symptom_index[symptom]:
                if similar == disease:
                    continue

                shared              = symptoms & disease_symptoms[similar]
                exclusive_to_similar = disease_symptoms[similar] - symptoms

                if len(shared) < min_shared_symptoms or not exclusive_to_similar:
                    continue

                absent_symptom = next(iter(exclusive_to_similar))
                candidates.append((similar, absent_symptom, len(shared)))

        candidates.sort(key=lambda x: -x[2])
        seen_similar = set()

        for similar, absent_symptom, shared_count in candidates:
            if similar in seen_similar:
                continue
            seen_similar.add(similar)

            test_cases.append({
                "disease":           disease,
                "present_symptoms":  list(symptoms),
                "absent_symptoms":   [absent_symptom],
                "expected_excluded": similar,
                "shared_count":      shared_count,
            })

            if len(seen_similar) >= max_cases_per_disease:
                break

    return test_cases


def evaluate_exclusion(test_cases, session):
    total = excl_correct = excl_wrong = survived = lost = no_match = 0
    details = []

    for case in test_cases:
        disease      = case["disease"]
        present      = case["present_symptoms"]
        absent       = case["absent_symptoms"]
        expected_excl = case["expected_excluded"]

        records     = run_query(session, present, absent)
        record_map  = {r["disease"]: r for r in records}

        our_rec  = record_map.get(disease)
        excl_rec = record_map.get(expected_excl)

        total += 1

        if our_rec is None and excl_rec is None:
            no_match += 1
            details.append({
                "disease":           disease,
                "expected_excluded":  expected_excl,
                "present_symptoms":  present,
                "absent_symptoms":   absent,
                "shared_count":      case["shared_count"],
                "our_passed":        None,
                "excl_passed":       None,
                "blocking_symptoms": [],
                "excl_excluded":     None,
                "our_survived":      None,
                "correct":           None,
            })
            continue

        our_passed  = our_rec["passed_filter"]  if our_rec  else None
        excl_passed = excl_rec["passed_filter"] if excl_rec else None

        excl_excluded  = (excl_rec is None) or (not excl_passed)
        
        our_survived_  = (our_rec is not None) and our_passed

        if excl_excluded:  excl_correct += 1
        else:              excl_wrong   += 1

        if our_survived_:  survived += 1
        else:              lost     += 1

        details.append({
            "disease":           disease,
            "expected_excluded":  expected_excl,
            "present_symptoms":  present,
            "absent_symptoms":   absent,
            "shared_count":      case["shared_count"],
            "our_passed":        our_passed,
            "excl_passed":       excl_passed,
            "blocking_symptoms": list(excl_rec["blocking_symptoms"]) if excl_rec else [],
            "excl_excluded":     excl_excluded,
            "our_survived":      our_survived_,
            "correct":           excl_excluded and our_survived_,
        })

    evaluated = total - no_match
    return {
        "total":              total,
        "evaluated":          evaluated,
        "no_match":           no_match,
        "excl_correct":       excl_correct,
        "excl_wrong":         excl_wrong,
        "survived":           survived,
        "lost":               lost,
        "exclusion_accuracy": round(excl_correct / evaluated * 100, 1) if evaluated else 0.0,
        "survival_accuracy":  round(survived     / evaluated * 100, 1) if evaluated else 0.0,
        "details":            details,
    }

## How the Exclusion Test Works

The exclusion test validates the inference layer's ability to correctly eliminate diseases based on symptoms the patient explicitly does not have.
For each disease in the test set, the algorithm identifies similar diseases — **those that share at least one symptom with it**. From each similar disease, a symptom is selected that the similar disease has but our disease does not. This symptom is then passed as an `absent_symptom` to the query.

The query is then executed with the full symptom list of our disease as `present_symptoms`, and the selected symptom as `absent_symptoms`. Since our disease does not have that symptom in its ontological profile, it should not be affected by the filter and should remain ranked. The similar disease, however, does have that symptom in its profile, so it should be blocked and appear with `passed_filter = false` in the results.
Each test case therefore has two expected outcomes:

1. **Our disease survives** — it remains in the ranked list with `passed_filter = true`, because none of its ontological symptoms appear in the absent list.
2. **The similar disease is excluded** — it is filtered out with `passed_filter = false`, because at least one of its ontological symptoms matches the absent symptom provided.

Two accuracy metrics are reported: `exclusion_accuracy`, measuring how often the similar disease is correctly blocked, and `survival_accuracy`, measuring how often our disease correctly remains in the results.

In [ ]:
excl_cases = build_exclusion_test_cases(
    ground_truth,
    min_shared_symptoms=1,
    max_cases_per_disease=3
)
print(f"Generated {len(excl_cases)} exclusion test cases.")

with driver.session() as session:
    excl = evaluate_exclusion(excl_cases, session)

e = excl["evaluated"]
print(f"{'='*50}")
print(f"EXCLUSION TEST")
print(f"{'='*50}")
print(f"Total test cases  : {excl['total']}")
print(f"Evaluated         : {e}")
print(f"No match          : {excl['no_match']}")
print(f"Excl. accuracy    : {excl['excl_correct']}/{e}  ({excl['exclusion_accuracy']}%)")
print(f"Survival accuracy : {excl['survived']}/{e}  ({excl['survival_accuracy']}%)")

from datetime import datetime

EXCL_RESULTS_PATH = BASE_DIR / "tests" / "inference-and-scoring-test" / "results" / "inference-exclusion-test.json"
EXCL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

output = {
    "timestamp": datetime.now().isoformat(),
    "config": {
        "min_match":           MIN_MATCH,
        "top_k":               TOP_K,
        "min_shared_symptoms": 1,
        "max_cases_per_disease": 3,
    },
    "summary": {
        "total":              excl["total"],
        "evaluated":          excl["evaluated"],
        "no_match":           excl["no_match"],
        "excl_correct":       excl["excl_correct"],
        "excl_wrong":         excl["excl_wrong"],
        "survived":           excl["survived"],
        "lost":               excl["lost"],
        "exclusion_accuracy": excl["exclusion_accuracy"],
        "survival_accuracy":  excl["survival_accuracy"],
    },
    "details": excl["details"]
}

with open(EXCL_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {EXCL_RESULTS_PATH}")